# Pick-time calibration

Tune the pick-time regression by **physical anchors (seconds)**, not raw coefficients.
The live curves use the *real* `Pick._pick_time`, so what you tune equals what the sim charges.

Formula: `pick_time = pick_intercept + qty*(pick_weight_coef*fn_w(w) + pick_volume_coef*fn_v(v)) + cart_swap_coef*swapped`  
Travel: `|dx|*x_speed + |dy|*y_speed`.

The weight/volume terms have a tunable **base function** `fn` (default `log`; also `linear` / `sqrt` / `pow:p`).

Drag the sliders until a typical pick is ~10x a single travel step and the per-quantity shape looks right; the derived coefficients print at the bottom, ready to paste into `REGRESSION_CONFIGS` / `PickConfig`.

Two playgrounds at the bottom: **Raw-coefficient sliders + sensitivity fan** (tune raw coefficients directly, with a fan showing ±2 variants of one coefficient) and the **Base-function family explorer** (compare `log/linear/sqrt/pow` for the weight/volume terms).

In [ ]:
import os, sys, math, pathlib
import numpy as np
import matplotlib.pyplot as plt

# find repo root (dir containing Warehouse/) walking up from cwd
p = pathlib.Path.cwd()
while p != p.parent and not (p / 'Warehouse').exists():
    p = p.parent
ROOT = str(p)
for sub in ('Warehouse', 'Optimization'):
    d = os.path.join(ROOT, sub)
    if d not in sys.path:
        sys.path.insert(0, d)

from Pick import PickConfig, _pick_time, DEFAULT_HEIGHT_BRACKETS   # the REAL sim formula
try:
    import ipywidgets as W
    _HAVE_WIDGETS = True
except Exception:
    _HAVE_WIDGETS = False
print('repo root:', ROOT, '| widgets:', _HAVE_WIDGETS)

# Objective: minimise the expected labor of a randomly-assigned task.  Target the
# regression so the reference task lands near this labor(handling) / travel split.
TARGET_LABOR_SHARE = 0.85        # handling : travel = 85 : 15

In [ ]:
# Reference archetypes (weight, volume) spanning the carton distribution, and the
# representative aisle TASK SHAPE that sets the handling/travel split.
ITEMS = {
    'light':   (3,    27),
    'typical': (20,   27_000),
    'heavy':   (55,   110_000),
    'bulky':   (15,   110_000),
}
W0, V0 = ITEMS['typical']        # anchor item for the per-unit handling anchor
COL_STEP = 48                    # pallet column physical step (Aisle_Dimensions)
LVL_STEP = 48                    # extra_large level step

# ── task shape — the knobs that decide the handling/travel split ───────────────
# handling = PICKS · (intercept + qty·handle)   ← placement-INVARIANT (the fixed cost)
# travel   = COLUMNS · walk_col + LEVELS · walk_level   ← the ONLY lever placement pulls
# These MUST match the real workload or the split below is fiction.  The sim's real tasks
# are far more handling-heavy than a toy 8-pick task (observed ~96% handling): many picks
# per aisle, little travel.  Use actual_travel_share(<run_dir>) below to read the realised
# split from a completed run, then set REF_TASK_PICKS / COLUMNS to reproduce it before
# solving for a new target.
REF_TASK_COLUMNS = 8             # distinct columns the picker sweeps in a task
REF_TASK_LEVELS  = 4             # vertical level movements in a task
REF_TASK_PICKS   = 8             # pick stops in the task
REF_TASK_QTY     = 2             # units per pick
LEVEL_WALK_RATIO = 0.6           # walk_sec_per_level / walk_sec_per_column (solver keeps this)

In [ ]:
def derive_coeffs(setup_sec, handle_sec_per_unit, weight_share,
                  walk_sec_per_column, walk_sec_per_level, cart_swap_sec):
    """Map physical anchors (seconds) -> raw PickConfig coefficients."""
    lnW, lnV = math.log(max(W0, 1)), math.log(max(V0, 1))
    pw = weight_share * handle_sec_per_unit / lnW if lnW else 0.0
    pv = (1 - weight_share) * handle_sec_per_unit / lnV if lnV else 0.0
    return dict(
        pick_intercept=setup_sec,
        pick_weight_coef=pw,
        pick_volume_coef=pv,
        x_speed=walk_sec_per_column / COL_STEP,
        y_speed=walk_sec_per_level / LVL_STEP,
        cart_swap_coef=cart_swap_sec,
    )

In [ ]:
def _fmt_brackets(bs):
    """Repr the height brackets with float('inf') (not bare 'inf') so the printed
    config block is valid Python to paste into REGRESSION_CONFIGS."""
    parts = ["(float('inf'), {!r})".format(m) if math.isinf(t) else "({!r}, {!r})".format(t, m)
             for t, m in bs]
    return "(" + ", ".join(parts) + ")"


def _fn_symbol(spec, var='w'):
    """Symbolic string for a transform spec, e.g. 'log' -> 'ln(w)', 'log:10' -> 'log10(w)',
    'pow:0.5' -> 'w^0.5', 'sqrt' -> 'sqrt(w)', 'linear' -> 'w'.  Used in the formula printouts."""
    if ':' in spec:
        name, p = spec.split(':', 1)
        if name == 'pow':  return f'{var}^{p}'
        if name == 'log':  return f'ln({var})' if p == 'e' else f'log{p}({var})'
        return f'{name}:{p}({var})'
    if spec == 'log':    return f'ln({var})'
    if spec == 'linear': return var
    if spec == 'sqrt':   return f'sqrt({var})'
    return f'{spec}({var})'


def render(setup_sec=3.0, handle_sec_per_unit=5.0, weight_share=0.7,
           walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30.0,
           config_name='calibrated',
           pick_intercept=None, pick_weight_coef=None, pick_volume_coef=None,
           x_speed=None, y_speed=None, cart_swap_coef=None):
    c = derive_coeffs(setup_sec, handle_sec_per_unit, weight_share,
                      walk_sec_per_column, walk_sec_per_level, cart_swap_sec)
    overrides = dict(pick_intercept=pick_intercept, pick_weight_coef=pick_weight_coef,
                     pick_volume_coef=pick_volume_coef, x_speed=x_speed,
                     y_speed=y_speed, cart_swap_coef=cart_swap_coef)
    for k, v in overrides.items():
        if v is not None:
            c[k] = v
    cfg = PickConfig(**c)

    qs = np.arange(1, 21)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    ax = axes[0]
    for name, (w, v) in ITEMS.items():
        ax.plot(qs, [_pick_time(cfg, w, v, int(q), False) for q in qs], marker='o', ms=3, label=name)
    ax.set_xlabel('quantity'); ax.set_ylabel('pick time (s)')
    ax.set_title('Pick time vs quantity (real _pick_time)'); ax.legend(); ax.grid(alpha=.3)

    ax = axes[1]
    ws = np.linspace(1, 60, 80)
    ax.plot(ws, [c['pick_weight_coef'] * math.log(max(w, 1)) for w in ws], label='weight term / unit')
    vs = np.linspace(27, 110_000, 80)
    ax.plot(ws, [c['pick_volume_coef'] * math.log(max(v, 1)) for v in vs], label='volume term / unit (vol swept)')
    ax.set_xlabel('weight (volume co-swept)'); ax.set_ylabel('s / unit')
    ax.set_title('Per-unit handling components'); ax.legend(fontsize=8); ax.grid(alpha=.3)

    ax = axes[2]
    w, v = ITEMS['typical']
    handling = REF_TASK_PICKS * _pick_time(cfg, w, v, REF_TASK_QTY, False)
    travel = REF_TASK_COLUMNS * COL_STEP * cfg.x_speed + REF_TASK_LEVELS * LVL_STEP * cfg.y_speed
    ax.bar(['travel', 'handling'], [travel, handling], color=['#4c72b0', '#dd8452'])
    ax.set_ylabel('time (s)'); ax.set_title('Reference task (~8 picks)'); ax.grid(axis='y', alpha=.3)
    # target labor-share line on the handling bar's expected height
    tgt_handling = TARGET_LABOR_SHARE / (1 - TARGET_LABOR_SHARE) * travel
    ax.axhline(tgt_handling, color='#c44e52', ls='--', lw=1,
               label=f'handling @ {TARGET_LABOR_SHARE:.0%} target')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

    one_pick = _pick_time(cfg, w, v, REF_TASK_QTY, False)
    one_step = COL_STEP * cfg.x_speed
    tot = travel + handling
    ratio = one_pick / one_step if one_step else float('inf')
    labor_share = handling / tot if tot else float('nan')
    print(f'typical pick (qty={REF_TASK_QTY}) = {one_pick:.1f}s | one column step = {one_step:.2f}s | pick:step = {ratio:.1f}x')
    print(f'reference task: handling {handling:.0f}s ({labor_share * 100:.0f}%) vs travel {travel:.0f}s ({(1 - labor_share) * 100:.0f}%)')
    print(f'  -> target labor share {TARGET_LABOR_SHARE:.0%}; current {labor_share:.0%} '
          f'({"OK" if abs(labor_share - TARGET_LABOR_SHARE) <= 0.02 else "tune handle/walk anchors"})')
    print()
    # ── how the formula actually works (symbolic, then with the current values) ──
    print('Formula (per pick, symbolic -> with current values):')
    print('  pick_time = M(y) * (intercept + qty*(pw*ln(w) + pv*ln(v))) + cart_swap*swapped')
    print('  travel    = |dx|*x_speed + |dy|*y_speed')
    print(f'  pick_time = M(y) * ({c["pick_intercept"]:.4g} + qty*({c["pick_weight_coef"]:.4g}*ln(w) '
          f'+ {c["pick_volume_coef"]:.4g}*ln(v))) + {c["cart_swap_coef"]:.4g}*swapped')
    print(f'  travel    = |dx|*{c["x_speed"]:.4g} + |dy|*{c["y_speed"]:.4g}')
    print(f'  M(y) = height bracket multiplier {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)}  (upper_y, mult)')
    print()
    print('# --- paste this entry into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---')
    print('    {')
    print(f"        'name'            : {config_name!r},")
    print(f"        'pick_intercept'  : {c['pick_intercept']:.6g},")
    print(f"        'pick_weight_coef': {c['pick_weight_coef']:.6g},")
    print(f"        'pick_volume_coef': {c['pick_volume_coef']:.6g},")
    print(f"        'cart_swap_coef'  : {c['cart_swap_coef']:.6g},")
    print(f"        'x_speed'         : {c['x_speed']:.6g},")
    print(f"        'y_speed'         : {c['y_speed']:.6g},")
    print(f"        'height_brackets' : {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)},")
    print('    },')
    return c

In [ ]:
# Interactive: anchors drive (sliders); derived raw coefficients print below the plots.
if _HAVE_WIDGETS:
    W.interact(
        render,
        setup_sec=W.FloatSlider(value=3.0, min=0, max=30, step=0.5, description='setup s/stop'),
        handle_sec_per_unit=W.FloatSlider(value=5.0, min=0, max=60, step=0.5, description='handle s/unit'),
        weight_share=W.FloatSlider(value=0.7, min=0, max=1, step=0.05, description='weight share'),
        walk_sec_per_column=W.FloatSlider(value=0.4, min=0.0, max=5, step=0.1, description='walk s/col'),
        walk_sec_per_level=W.FloatSlider(value=0.25, min=0.0, max=5, step=0.1, description='walk s/level'),
        cart_swap_sec=W.FloatSlider(value=30.0, min=0, max=60*10, step=20, description='cart swap s'),
        config_name=W.Text(value='calibrated', description='config name'),
        pick_intercept=W.fixed(None), pick_weight_coef=W.fixed(None), pick_volume_coef=W.fixed(None),
        x_speed=W.fixed(None), y_speed=W.fixed(None), cart_swap_coef=W.fixed(None),
    )
else:
    render()

## Travel-share tools

Handling per task is **placement-invariant** (`intercept + qty·handle`, summed over picks) — you can't slot your way out of it. **Travel is the only lever placement can pull**, so the regression has to give travel a real share or no assignment policy can separate.

- **`solve_walk_for_travel_share(target, …)`** — back-computes the walk speeds (`x_speed`/`y_speed`) needed to hit a target travel share for the reference task, then renders + prints a paste-ready config.
- **`actual_travel_share(run_dir)`** — reads the *realised* travel share from a completed run. The reference task is only honest if `REF_TASK_PICKS`/`COLUMNS`/`LEVELS`/`QTY` reproduce it, so **check a real run first, align the task-shape constants, then solve.**

In [ ]:
# ── TOOL 1 — solve walk speeds for a target TRAVEL share ───────────────────────
# Handling per task is placement-invariant; travel is the lever.  Hold the handling anchors
# + task shape fixed and back-solve walk_sec_per_column/level to hit a target travel share,
# then render + print a paste-ready config.
def solve_walk_for_travel_share(target_travel_share=1 - TARGET_LABOR_SHARE,
                                setup_sec=30.0, handle_sec_per_unit=10.0,
                                weight_share=0.7, cart_swap_sec=30.0,
                                config_name='calibrated'):
    s = target_travel_share
    handling = REF_TASK_PICKS * (setup_sec + REF_TASK_QTY * handle_sec_per_unit)
    travel_needed = s / (1 - s) * handling
    walk_col   = travel_needed / (REF_TASK_COLUMNS + LEVEL_WALK_RATIO * REF_TASK_LEVELS)
    walk_level = LEVEL_WALK_RATIO * walk_col
    # ── how the solve works (symbolic, then with the current values) ──
    print('Formula (solve, symbolic -> with current values):')
    print('  handling      = PICKS*(setup + QTY*handle)')
    print('  travel_needed = s/(1-s) * handling')
    print('  walk_col      = travel_needed / (COLUMNS + ratio*LEVELS) ;  walk_level = ratio*walk_col')
    print(f'  handling      = {REF_TASK_PICKS}*({setup_sec:g} + {REF_TASK_QTY}*{handle_sec_per_unit:g}) = {handling:.0f}')
    print(f'  travel_needed = {s:g}/(1-{s:g}) * {handling:.0f} = {travel_needed:.0f}')
    print(f'  walk_col      = {travel_needed:.0f} / ({REF_TASK_COLUMNS} + {LEVEL_WALK_RATIO:g}*{REF_TASK_LEVELS}) = {walk_col:.3f}')
    print()
    print(f'target travel share {s:.0%}  ->  walk_sec_per_column={walk_col:.3f}  '
          f'walk_sec_per_level={walk_level:.3f}')
    print(f'(reference task: handling={handling:.0f}s, travel target={travel_needed:.0f}s)\n')
    return render(setup_sec=setup_sec, handle_sec_per_unit=handle_sec_per_unit,
                  weight_share=weight_share, walk_sec_per_column=walk_col,
                  walk_sec_per_level=walk_level, cart_swap_sec=cart_swap_sec,
                  config_name=config_name)

# ── TOOL 2 — read the ACTUAL travel share from a completed run ─────────────────
# Aligns the reference task to reality: reads per-strategy travel_time/handling_time from a
# run's stats_summary.csv (needs run_analysis to have produced stats/).  Set REF_TASK_PICKS /
# COLUMNS so render's split matches this before trusting solve_walk_for_travel_share.
def actual_travel_share(run_dir, config='calibrated'):
    import glob, csv
    tt, ht = [], []
    for f in glob.glob(os.path.join(run_dir, '*', config, 'stats', 'stats_summary.csv')):
        with open(f, encoding='utf-8') as fh:
            for r in csv.DictReader(fh):
                if r['metric'] == 'travel_time':   tt.append(float(r['mean']))
                if r['metric'] == 'handling_time': ht.append(float(r['mean']))
    if not tt:
        print('no stats_summary.csv under', run_dir, '(run run_analysis with stats first)')
        return None
    T, H = sum(tt) / len(tt), sum(ht) / len(ht)
    share = T / (T + H)
    print(f'{config}: actual travel share = {share:.1%}   '
          f'(travel {T:,.0f} / handling {H:,.0f}, n={len(tt)})')
    return share

# Example: dial the regression to 15% travel for the reference task (edit & run)
_ = solve_walk_for_travel_share(target_travel_share=0.15, setup_sec=30.0, handle_sec_per_unit=10.0)
# Example: check what a real run actually produced (edit the path, then set REF_TASK_* to match):
# actual_travel_share(r'H:\Data\Inventory_Optimizer_Data\Optimization_Outputs\comparison_20260619_172724')

### Override a raw coefficient directly
The sliders are physical anchors; to hand-tune a raw coefficient, call `render` with it explicitly (overrides the derived value), e.g.:
```python
render(setup_sec=3, handle_sec_per_unit=5, weight_share=0.7,
       walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30,
       pick_weight_coef=1.6)   # <- raw override
```

In [ ]:
# scratch: direct raw-coefficient override example (edit + run)
_ = render(setup_sec=30.0, handle_sec_per_unit=10.0, weight_share=0.7,
           walk_sec_per_column=0.4, walk_sec_per_level=0.25, cart_swap_sec=30.0)

## Within-aisle travel-share calibrator

Travel speed is **grounded by equipment** (1 s/column, 1.5 s/level) and **held fixed**. You instead set the within-aisle **task geometry** — number of *picks*, number of *pick locations* (travel legs), and the *Manhattan distance per leg* (in column-steps, with a vertical fraction) — and tune the **handling anchors** (setup s/pick, handle s/unit) until the travel share lands in the **15–20% band** (shown in green). Match `picks` / `pick locs` / `manhattan` to a real run (via `actual_travel_share` above) so the split reflects your actual workload, then read off the paste-ready config.

In [ ]:
# ── Within-aisle travel-share calibrator ──────────────────────────────────────
# Travel speeds are GROUNDED by equipment and held FIXED (1 s/column, 1.5 s/level).
# You set the within-aisle TASK GEOMETRY — picks, pick locations, and the Manhattan
# travel distance per leg (in column-steps, with a vertical fraction since y is 1.5x x)
# — and tune the HANDLING anchors (setup s/pick, handle s/unit) until travel lands in the
# 15-20% band.  Match picks/locations/manhattan to a real run (actual_travel_share above)
# so the split is real, not a toy.
WALK_SEC_PER_COLUMN = 1.0     # GROUNDED (equipment) — do not change
WALK_SEC_PER_LEVEL  = 1.5     # GROUNDED (equipment) — do not change

def within_aisle_split(picks=20, pick_locations=15, manhattan_dist=8.0, frac_vertical=0.3,
                       setup_sec=15.0, handle_sec_per_unit=3.0, qty=2.0,
                       weight_share=0.7, config_name='calibrated'):
    """Within-aisle handling/travel split. Travel = pick_locations legs, each of
    `manhattan_dist` column-steps at the grounded speed (a fraction `frac_vertical`
    of each step is vertical @1.5 s, the rest horizontal @1 s).  Handling = picks ×
    (setup + qty·handle).  Speeds are fixed; tune the handling anchors to hit ~15-20%."""
    per_pick  = setup_sec + qty * handle_sec_per_unit
    eff_step  = (1 - frac_vertical) * WALK_SEC_PER_COLUMN + frac_vertical * WALK_SEC_PER_LEVEL
    per_leg   = manhattan_dist * eff_step
    handling  = picks * per_pick
    travel    = pick_locations * per_leg
    tot       = handling + travel
    share     = travel / tot if tot else 0.0

    fig, ax = plt.subplots(figsize=(7, 1.5))
    ax.barh([0], [travel],  color='#4c72b0', label=f'travel {travel:.0f}s')
    ax.barh([0], [handling], left=[travel], color='#dd8452', label=f'handling {handling:.0f}s')
    ax.axvline(0.15 * tot, color='g', ls='--', lw=1)
    ax.axvline(0.20 * tot, color='g', ls='--', lw=1, label='15-20% target')
    ax.set_yticks([]); ax.set_xlabel('within-aisle task labor (s)')
    ax.set_title(f'travel share = {share*100:.1f}%   (green = 15-20% target band)')
    ax.legend(fontsize=8, loc='center right'); plt.tight_layout(); plt.show()

    flag = ('OK' if 0.15 <= share <= 0.20 else
            ('travel LOW  -> lower handling' if share < 0.15 else 'travel HIGH -> raise handling'))
    print(f'travel share = {share*100:.1f}%   [{flag}]')
    print(f'  handling: {picks} picks x {per_pick:.1f} s/pick (setup {setup_sec} + {qty}x{handle_sec_per_unit}) = {handling:.0f} s')
    print(f'  travel  : {pick_locations} legs x {per_leg:.2f} s/leg ({manhattan_dist} steps, {frac_vertical:.0%} vertical, GROUNDED 1/1.5 s) = {travel:.0f} s')
    print()
    # ── how the split formula works (symbolic, then with the current values) ──
    print('Formula (symbolic -> with current values):')
    print('  handling = picks*(setup + qty*handle)')
    print('  travel   = locations*manhattan*((1-vf)*col_sec + vf*lvl_sec)')
    print('  share    = travel / (handling + travel)')
    print(f'  handling = {picks}*({setup_sec:g} + {qty:g}*{handle_sec_per_unit:g}) = {handling:.0f}')
    print(f'  travel   = {pick_locations}*{manhattan_dist:g}*((1-{frac_vertical:g})*{WALK_SEC_PER_COLUMN:g} '
          f'+ {frac_vertical:g}*{WALK_SEC_PER_LEVEL:g}) = {travel:.0f}')
    print(f'  share    = {travel:.0f} / ({handling:.0f} + {travel:.0f}) = {share*100:.1f}%')
    print()
    c = derive_coeffs(setup_sec, handle_sec_per_unit, weight_share,
                      WALK_SEC_PER_COLUMN, WALK_SEC_PER_LEVEL, cart_swap_sec=300.0)
    print('# --- paste into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---')
    print('    {')
    print(f"        'name'            : {config_name!r},")
    print(f"        'pick_intercept'  : {c['pick_intercept']:.6g},")
    print(f"        'pick_weight_coef': {c['pick_weight_coef']:.6g},")
    print(f"        'pick_volume_coef': {c['pick_volume_coef']:.6g},")
    print(f"        'cart_swap_coef'  : {c['cart_swap_coef']:.6g},")
    print(f"        'x_speed'         : {c['x_speed']:.6g},")
    print(f"        'y_speed'         : {c['y_speed']:.6g},")
    print(f"        'height_brackets' : {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)},")
    print('    },')
    return share

if _HAVE_WIDGETS:
    W.interact(
        within_aisle_split,
        picks=W.IntSlider(value=20, min=1, max=60, description='picks'),
        pick_locations=W.IntSlider(value=15, min=1, max=60, description='pick locs'),
        manhattan_dist=W.FloatSlider(value=8.0, min=0, max=50, step=0.5, description='manhattan/leg'),
        frac_vertical=W.FloatSlider(value=0.3, min=0, max=1, step=0.05, description='vert frac'),
        setup_sec=W.FloatSlider(value=15.0, min=0, max=60, step=1, description='setup s/pick'),
        handle_sec_per_unit=W.FloatSlider(value=3.0, min=0, max=20, step=0.5, description='handle s/unit'),
        qty=W.FloatSlider(value=2.0, min=1, max=10, step=0.5, description='qty/pick'),
        weight_share=W.FloatSlider(value=0.7, min=0, max=1, step=0.05, description='weight share'),
        config_name=W.Text(value='calibrated', description='config name'),
    )
else:
    within_aisle_split()

## Raw-coefficient sliders + sensitivity fan

Tune the **raw `PickConfig` coefficients directly** (bypassing the physical-anchor derivation) and watch the real `_pick_time` curve move. The **`fan`** dropdown sweeps one chosen coefficient ±2 steps within a **`ratio`** range (e.g. `1.5` = ±50%): the chosen value is a **bold dark** line, the four variants are **lighter** lines at geometric factors `[1/ratio, 1/√ratio, √ratio, ratio]`. Every line is computed with the real `_pick_time`, so the fan shows the actual sim charge, not an approximation. The reference-task split + paste-ready config print below for the chosen values.

In [ ]:
# ── Raw-coefficient sliders + sensitivity fan ─────────────────────────────────
def render_raw(pick_weight_coef=2.33666, pick_volume_coef=0.294014, pick_intercept=30.0,
               x_speed=0.141403, y_speed=0.0848416, cart_swap_coef=30.0,
               fan='pick_weight_coef', ratio=1.5, config_name='calibrated'):
    """Tune raw coefficients directly. The pick-time-vs-qty plot shows the chosen config
    as a bold dark line plus a sensitivity FAN: the `fan` coefficient swept 2 below + 2
    above (geometric in `ratio`) as lighter lines.  All lines use the real _pick_time."""
    base = dict(pick_intercept=pick_intercept, pick_weight_coef=pick_weight_coef,
                pick_volume_coef=pick_volume_coef, x_speed=x_speed, y_speed=y_speed,
                cart_swap_coef=cart_swap_coef)
    cfg = PickConfig(**base)
    qs  = np.arange(1, 21)
    w, v = ITEMS['typical']

    fig, ax = plt.subplots(figsize=(8, 4.6))
    # fan: 2 below + 2 above the chosen coefficient, geometric in `ratio`
    factors = [1 / ratio, 1 / math.sqrt(ratio), math.sqrt(ratio), ratio]
    for f in factors:
        d = dict(base); d[fan] = base[fan] * f
        cfgf = PickConfig(**d)
        ax.plot(qs, [_pick_time(cfgf, w, v, int(q), False) for q in qs],
                color='#888888', lw=1.0, alpha=0.35, label=f'{fan} ×{f:.2f}')
    # chosen — bold dark, on top
    ax.plot(qs, [_pick_time(cfg, w, v, int(q), False) for q in qs],
            color='#111111', lw=2.5, marker='o', ms=3, label='chosen', zorder=5)
    ax.set_xlabel('quantity'); ax.set_ylabel('pick time (s)')
    ax.set_title(f"Pick time vs quantity (typical) — fan: {fan} within ×[{1/ratio:.2f}, {ratio:.2f}]")
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

    # reference-task split + paste-ready config for the CHOSEN values (reuse render's overrides)
    return render(setup_sec=0.0, handle_sec_per_unit=0.0, weight_share=0.7,
                  walk_sec_per_column=x_speed * COL_STEP, walk_sec_per_level=y_speed * LVL_STEP,
                  cart_swap_sec=cart_swap_coef, config_name=config_name,
                  pick_intercept=pick_intercept, pick_weight_coef=pick_weight_coef,
                  pick_volume_coef=pick_volume_coef, x_speed=x_speed, y_speed=y_speed,
                  cart_swap_coef=cart_swap_coef)

if _HAVE_WIDGETS:
    W.interact(
        render_raw,
        pick_weight_coef=W.FloatSlider(value=2.33666, min=0, max=6, step=0.02, description='weight coef'),
        pick_volume_coef=W.FloatSlider(value=0.294014, min=0, max=1.5, step=0.005, description='volume coef'),
        pick_intercept=W.FloatSlider(value=30.0, min=0, max=60, step=1, description='intercept'),
        x_speed=W.FloatSlider(value=0.141403, min=0, max=0.4, step=0.001, readout_format='.4f', description='x_speed'),
        y_speed=W.FloatSlider(value=0.0848416, min=0, max=0.4, step=0.001, readout_format='.4f', description='y_speed'),
        cart_swap_coef=W.FloatSlider(value=30.0, min=0, max=600, step=10, description='cart swap'),
        fan=W.Dropdown(options=['pick_weight_coef', 'pick_volume_coef', 'pick_intercept'],
                       value='pick_weight_coef', description='fan coef'),
        ratio=W.FloatSlider(value=1.5, min=1.05, max=3.0, step=0.05, description='fan ratio'),
        config_name=W.Text(value='calibrated', description='config name'),
    )
else:
    render_raw()

## Base-function family explorer

The weight and volume handling terms don't have to be `ln`. This section prototypes the **composable** handling term — each term = `coef · fn(value)` where `fn` ∈ `log[:base] / linear / sqrt / pow:p` — mirroring the planned `cost_model.LaborModel`. The `log` family's **base is selectable** (`e` / `2` / `10`, emitted as `log:<base>`); note a non-`e` base only rescales the term by `1/ln(base)`, i.e. it's equivalent to a smaller coefficient. The first two panels overlay each candidate base function for the weight and volume terms (selected = **bold dark**, others lighter; carton archetypes marked); the third shows end-to-end pick-time-vs-qty for the **selected pair** vs the `ln/ln` baseline. The paste-ready block additionally emits `pick_weight_fn` / `pick_volume_fn` (forward-compatible with `LaborModel.from_dict`).

In [ ]:
# ── Base-function family explorer ─────────────────────────────────────────────
# Local prototype of the composable handling term (mirrors planned cost_model.LaborModel):
# each term = coef * fn(value), fn in {log[:base], linear, sqrt, pow:p}.  Weight and volume
# carry their OWN base/exponent, so the two terms compose independently.
TRANSFORMS = {
    'log':    lambda v: math.log(max(v, 1.0)),   # natural log (base e)
    'linear': lambda v: float(v),
    'sqrt':   lambda v: math.sqrt(max(v, 0.0)),
}

def resolve_transform(spec):
    """'name' or 'name:param' -> callable.
    e.g. 'pow:0.5' -> v**0.5 ; 'log' -> ln ; 'log:10' -> log base 10 ; 'log:2' -> log base 2."""
    if ':' in spec:
        name, p = spec.split(':', 1)
        if name == 'pow':
            p = float(p)
            return lambda v: max(v, 0.0) ** p
        if name == 'log':
            if p == 'e':
                return TRANSFORMS['log']
            lnb = math.log(float(p))            # log_b(x) = ln(x) / ln(b)
            return lambda v: math.log(max(v, 1.0)) / lnb
        raise ValueError(f'unknown parametric transform {name!r}')
    return TRANSFORMS[spec]

def term_spec(fam, pow_p=0.5, log_base='e'):
    """Compose a family + its parameter into a transform spec the composable functions accept."""
    if fam == 'pow':  return f'pow:{pow_p:g}'
    if fam == 'log':  return 'log' if log_base == 'e' else f'log:{log_base}'
    return fam

def handle_var_fn(w, v, wc, vc, wfn, vfn):
    return wc * resolve_transform(wfn)(w) + vc * resolve_transform(vfn)(v)

def per_pick_fn(w, v, qty, intercept, wc, vc, wfn, vfn):
    """Ground-level per pick (M=1, no cart): intercept + qty*handle_var.  Mirrors the
    planned LaborModel.per_pick at ground level.  wfn/vfn are full specs (incl. base/exp)."""
    return intercept + qty * handle_var_fn(w, v, wc, vc, wfn, vfn)

_FAMILIES = ['log', 'linear', 'sqrt', 'pow']   # 'pow' uses *_pow_p; 'log' uses *_log_base

def compare_transforms(pick_weight_coef=2.33666, pick_volume_coef=0.294014, pick_intercept=30.0,
                       weight_fn='log', weight_pow_p=0.5, weight_log_base='e',
                       volume_fn='log', volume_pow_p=0.5, volume_log_base='e',
                       config_name='calibrated'):
    wsel = term_spec(weight_fn, weight_pow_p, weight_log_base)
    vsel = term_spec(volume_fn, volume_pow_p, volume_log_base)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    def _term_panel(ax, coef, xs, selected_fam, pow_p, log_base, label_axis):
        for fam in _FAMILIES:
            sp  = term_spec(fam, pow_p, log_base)
            f   = resolve_transform(sp)
            sel = (fam == selected_fam)
            ax.plot(xs, [coef * f(x) for x in xs],
                    lw=2.6 if sel else 1.0, alpha=1.0 if sel else 0.35,
                    color='#111111' if sel else None, zorder=5 if sel else 1,
                    label=sp + (' (selected)' if sel else ''))
        # keep curvature readable: clip y to the non-linear families (linear may shoot off-top)
        nonlin = [coef * resolve_transform(term_spec(fm, pow_p, log_base))(xs[-1])
                  for fm in ('log', 'sqrt', 'pow')]
        top = max(nonlin) * 1.4
        if top > 0:
            ax.set_ylim(0, top)
        ax.set_xlabel(label_axis); ax.set_ylabel('term (s/unit)')
        ax.legend(fontsize=8); ax.grid(alpha=.3)

    ws = np.linspace(1, 60, 120)
    _term_panel(axes[0], pick_weight_coef, ws, weight_fn, weight_pow_p, weight_log_base, 'weight')
    axes[0].set_title('Weight term per base function')
    for _nm, (w, _v) in ITEMS.items():
        axes[0].axvline(w, color='gray', ls=':', lw=0.6)

    vs = np.linspace(27, 110_000, 120)
    _term_panel(axes[1], pick_volume_coef, vs, volume_fn, volume_pow_p, volume_log_base, 'volume')
    axes[1].set_title('Volume term per base function (linear clipped)')

    # end-to-end: selected pair (dark) vs ln/ln baseline (light dashed)
    ax = axes[2]
    qs = np.arange(1, 21)
    w, v = ITEMS['typical']
    ax.plot(qs, [per_pick_fn(w, v, int(q), pick_intercept, pick_weight_coef, pick_volume_coef, 'log', 'log') for q in qs],
            color='#999999', lw=1.5, ls='--', label='ln/ln baseline')
    ax.plot(qs, [per_pick_fn(w, v, int(q), pick_intercept, pick_weight_coef, pick_volume_coef, wsel, vsel) for q in qs],
            color='#111111', lw=2.6, marker='o', ms=3, label=f'{wsel} / {vsel}')
    ax.set_xlabel('quantity'); ax.set_ylabel('pick time (s, ground)')
    ax.set_title('Pick time vs qty (typical): selected vs baseline')
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); plt.show()

    print(f'selected base functions: weight={wsel}  volume={vsel}')
    for nm, fam, lb in (('weight', weight_fn, weight_log_base), ('volume', volume_fn, volume_log_base)):
        if fam == 'log' and lb != 'e':
            print(f'  note: {nm} log base {lb} rescales its coef by 1/ln({lb})={1/math.log(float(lb)):.4g} '
                  f'(log_b(x)=ln(x)/ln(b)) — equivalent to a smaller coefficient.')
    print()
    # ── how the composable formula works (symbolic, then with the current values) ──
    print('Formula (ground per pick, symbolic -> with current values):')
    print('  per_pick   = intercept + qty*(pw*fn_w(weight) + pv*fn_v(volume))')
    print('  handle_var = pw*fn_w(weight) + pv*fn_v(volume)')
    print(f'  per_pick   = {pick_intercept:.4g} + qty*({pick_weight_coef:.4g}*{_fn_symbol(wsel, "w")} '
          f'+ {pick_volume_coef:.4g}*{_fn_symbol(vsel, "v")})')
    print()
    print('# --- paste into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---')
    print('    {')
    print(f"        'name'            : {config_name!r},")
    print(f"        'pick_intercept'  : {pick_intercept:.6g},")
    print(f"        'pick_weight_coef': {pick_weight_coef:.6g},")
    print(f"        'pick_weight_fn'  : {wsel!r},")
    print(f"        'pick_volume_coef': {pick_volume_coef:.6g},")
    print(f"        'pick_volume_fn'  : {vsel!r},")
    print(f"        'height_brackets' : {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)},")
    print('    },')

if _HAVE_WIDGETS:
    W.interact(
        compare_transforms,
        pick_weight_coef=W.FloatSlider(value=2.33666, min=0, max=6, step=0.02, description='weight coef'),
        pick_volume_coef=W.FloatSlider(value=0.294014, min=0, max=1.5, step=0.005, description='volume coef'),
        pick_intercept=W.FloatSlider(value=30.0, min=0, max=60, step=1, description='intercept'),
        weight_fn=W.Dropdown(options=_FAMILIES, value='log', description='weight fn'),
        weight_pow_p=W.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='weight pow p'),
        weight_log_base=W.Dropdown(options=['e', '2', '10'], value='e', description='weight log base'),
        volume_fn=W.Dropdown(options=_FAMILIES, value='log', description='volume fn'),
        volume_pow_p=W.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='volume pow p'),
        volume_log_base=W.Dropdown(options=['e', '2', '10'], value='e', description='volume log base'),
        config_name=W.Text(value='calibrated', description='config name'),
    )
else:
    compare_transforms()